# Mechanism algorithms: quota, subsidy, penalty and social influence

This tutorial teaches the four mechanisms shipped in
`core.mechanism.algorithms`, the mathematics that each one implements, and the way
they compose when one regulator holds several of them. The benchmark is the
fishery of `examples/bilevel_fishery`. Every formula is stated next to the code
that implements it and is then checked by calling the mechanism on concrete
inputs, so that you can see the equation at work on numbers before you meet it in
a training run.

The notebook runs from top to bottom in well under a minute. Its last part builds
the complete bilevel configuration with the present builder, trains a tiny
version of it, and points to the commands that launch the full experiment.

## Setup

The `examples/` directory is not an installed package, so the first cell puts the
repository root on the import path before anything is imported from it.

In [ ]:
import sys
from pathlib import Path

# `examples/` is not an installed package: put the repository root on the path.
REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import dataclasses
import math
import tempfile
from types import SimpleNamespace
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import ray
from gymnasium import spaces
from gymnasium.spaces import unflatten
from scipy.special import expit

from core.adaptors.ray.marl_env import RLlibMultiAgentEnvAdapter
from core.agents.base import AgentConfig
from core.callbacks import log_and_report_episode_metrics, tag_episode_with_env_idx
from core.envs import marl_regulated
from core.mechanism.algorithms.penalty import ThresholdPenalty
from core.mechanism.algorithms.quota import Quota
from core.mechanism.algorithms.social_influence import SocialInfluence
from core.mechanism.algorithms.subsidy import MAX_SUBSIDY, Subsidy
from core.mechanism.base import MDPState
from core.optimizers.appo.config import APPOptimizerConfig
from core.optimizers.bilevel import BilevelConfig
from core.optimizers.es.config import ESConfig
from core.optimizers.es.optimizer import ESOptimizer
from core.reporting.csv import CSVConfig
from core.utils import ACTION_TEMPERATURE
from core.world.context import MechanismContext, MechanismStatus
from examples.bilevel_fishery.metric_schema import FisheryMetricSchema
from examples.bilevel_fishery.queries import ES_QUERIES, INNER_QUERIES
from examples.bilevel_fishery.regulated_env import (
    FishermanConfig,
    FisheryRegulatedEnv,
    FishingConfig,
    RestoreConfig,
)
from examples.bilevel_fishery.regulator_env import FisheryRegulatorEnv

# 1. Reinforcement learning before mechanisms

The agents of the benchmark do not see the state of the world but an observation
of it, so the decision problem is strictly a partially observed Markov decision
process. To keep the notation light, this tutorial writes the dynamics and the
Bellman equation on the state $s_t$, as for a fully observed process, and brings
the observation $o_t$ in only where a mechanism acts on it. The environment is
the tuple

$$
\mathcal E = (\mathcal S,\mathcal A,P,R,\gamma).
$$

At each time $t$ the environment is in a state $s_t$, the agent receives an
observation $o_t$, its policy samples an action

$$
a_t \sim \pi_\phi(\cdot\mid o_t),
$$

the environment moves to a new state $s_{t+1}\sim P(\cdot\mid s_t,a_t)$, and the
agent receives a reward $r_t$. The value of a policy satisfies the Bellman
equation

$$
V^{\pi_\phi}(s_t)
=
\mathbb E\left[r_t+\gamma V^{\pi_\phi}(s_{t+1})\right].
$$

The learned policy depends on the experience that this loop generates. Changing
what the agents observe, what their actions do, or what they are paid therefore
changes the optimization problem they solve, and this is the lever that a
mechanism pulls.

# 2. Mechanism intervention

In the present framework a mechanism is owned by a leader, the regulator, and
acts on the followers, the learning agents. Its parameters $\theta$ are the
regulator's own action: the outer optimizer searches them, and the inner
optimizer trains the followers under them. A mechanism never overwrites the
shared state. It reads the state and returns a **residual**, a partial state that
holds only the fields it changes, and the owning agent adds the residuals of all
its mechanisms to the state.

Write $\Delta_k$ for the residual of mechanism $k$, with parameter $\theta_k$.
Three channels can carry a residual. On the action channel, the raw policy output
$z_{i,t}$ of follower $i$ is moved before the follower's own mechanisms decode
it:

$$
z_{i,t}^{*} = z_{i,t} + \sum_k \Delta^{A}_{k,i}(x_t;\theta_k).
$$

On the reward channel, the reward is shifted:

$$
r_{i,t}^{*} = r_{i,t} + \sum_k \Delta^{R}_{k,i}(x_t;\theta_k).
$$

On the observation channel, a contribution is added to the observation that the
policy receives at the next step:

$$
o_{i,t+1}^{*} = o_{i,t+1} + \sum_k \Delta^{O}_{k,i}(x_{t+1};\theta_k).
$$

Here $x_t$ is the shared state at step $t$, which holds the raw actions of all
agents. The learner then optimizes the same Bellman equation as before, with
$r^{*}$ in place of $r$. The shaped observation $o^{*}$ takes the place of $o$ as
the input of the policy $\pi_\phi(\cdot\mid o^{*}_t)$, and the equation keeps
its form on the state:

$$
V^{\pi_\phi,\theta}(s_t)
=
\mathbb E\left[r_t^{*}+\gamma V^{\pi_\phi,\theta}(s_{t+1})\right].
$$

Because every residual is added to the state, the mechanism does not need to
know what the other mechanisms of its agent do, and the benchmark does not need
to know that a regulator exists. This separation lets a benchmark describe the
ecology and physics while a mechanism describes the regulation.

## 2.1 How this maps to code

A mechanism is built from a frozen configuration object with
`MechanismConfig.build(aid)`, where `aid` is the identifier of the owning agent.
The `Mechanism` base class has three methods that a subclass implements or
overrides, and a `__call__` that chains the first two. `decode(mdp, action)` maps the
regulator's raw action to the coordinates of the mechanism, for example by
clipping it to its valid range. `apply(mdp, action)` receives the decoded action
and returns the residual of the transition. `observe(mdp)` returns the residual
of the observation channel, and by default contributes nothing. Calling the
mechanism object, `__call__`, runs `decode`, records the decoded action in the
shared state, and runs `apply`.

Two configuration fields connect a mechanism to a benchmark without hard-coding
it. `acts_on=(agent, mechanism)` names the followers that the mechanism targets:
the agent type, such as `"fisherman"`, and the identifier of the follower
mechanism whose action it changes, such as `"harvest"`. `obs_map` gives the names
under which the mechanism reads the state: `{"resource_level": "fish"}` says that
the resource level is the `"fish"` entry of the state, and the mechanism divides
it by the carrying capacity `mdp.params["K"]` that the environment publishes. The
same algorithm can therefore regulate fish biomass, water in a reservoir, or any
other normalised resource.

The cell below builds the quota of the canonical fishery experiment and shows
these fields. The meaning of each argument is explained in section 4.

In [ ]:
UNIT_BOX = spaces.Box(low=0.0, high=1.0, shape=(1,), dtype=np.float32)
UNBOUNDED_BOX = spaces.Box(low=-np.inf, high=np.inf, shape=(1,), dtype=np.float32)

quota_cfg = Quota(
    id="quota",
    action_space=UNIT_BOX,
    acts_on=("fisherman", "harvest"),
    obs_map={"resource_level": "fish"},
    quota_transition_width=0.03,
    usage_transition_width=0.005,
    default=np.asarray(0.56224, dtype=np.float32),
)
quota = quota_cfg.build("regulator")

print("class:", type(quota).__name__, "| owner:", quota.aid, "| id:", quota.id)
print("acts on (agent type, mechanism id):", quota.acts_on)
print("reads the resource level through:", quota.obs_map)
print("regulator action space:", quota.action_space)
print("decode(1.7) clips an out-of-range action:", quota.decode(None, np.array([1.7])))

# 3. Fishery benchmark mathematics

The benchmark is a single fish stock shared by $N$ fishers. Let $B_t$ be the fish
biomass. The stock follows the discrete-time surplus-production model of
Schaefer (1954) and Hilborn and Walters (1992), with the Pella and Tomlinson
(1969) production function

$$
g(B_t)
=
\frac{r}{p}\,B_t
\left[
1-\left(\frac{B_t}{K}\right)^p
\right],
$$

where $r$ is the intrinsic growth rate, $K$ the carrying capacity, and $p$ a
shape parameter. The case $p=1$ is the Schaefer logistic model. The stock
evolves as

$$
B_{t+1}
=
\max\left(B_t+g(B_t)+R_t+\xi_t-C_t,\;0\right),
$$

where $R_t$ is the total restoration of the fishers, $C_t$ the total catch that
is actually delivered, and $\xi_t=\sigma\,\varepsilon_t B_t$ with
$\varepsilon_t\sim\mathcal N(0,1)$ is process noise proportional to the stock.
The reference points follow from $g'(B)=0$:

$$
B_{\mathrm{MSY}}=K\left(\tfrac{1}{p+1}\right)^{1/p},
\qquad
\mathrm{MSY}=\frac{rK}{(p+1)^{(p+1)/p}},
\qquad
F_{\mathrm{MSY}}=\frac{\mathrm{MSY}}{B_{\mathrm{MSY}}}.
$$

Each fisher $i$ has two raw policy outputs, $z^{h}_{i,t}$ for harvest and
$z^{e}_{i,t}$ for restoration. The benchmark maps each one to a fraction in
$(0,1)$ with the logistic function $s(x)=1/(1+e^{-x})$ at temperature $4$,

$$
h_{i,t}=s\!\left(\frac{z^{h}_{i,t}}{4}\right),
\qquad
e_{i,t}=s\!\left(\frac{z^{e}_{i,t}}{4}\right),
$$

where the harvest fraction $h_{i,t}$ scales the catch the fisher requests,

$$
Q_{i,t}=h_{i,t}\,m\,F_{\mathrm{MSY}}\frac{B_t}{N},
$$

with $m$ the `unregulated_f_multiplier`, and the effort $e_{i,t}$ scales the
restoration, $R_t=\rho K\sum_i e_{i,t}/N$, with $\rho$ the
`restoration_effectiveness`. The temperature $4$ is a heuristic scale of the
benchmark (`core.utils.ACTION_TEMPERATURE`), not a value taken from the
literature.

All fishers act at the same time on the stock $B_t$ that the step starts with.
If the total request $Q_t=\sum_i Q_{i,t}$ exceeds $B_t$, every fisher receives the
same share of its request, so that the catch never exceeds the stock:

$$
C_t = \min\left(1,\frac{B_t}{Q_t}\right)Q_t.
$$

The fisher's reward is the harvest fraction $h^{*}_{i,t}$ it was left with after
the regulator's mechanisms acted, computed before this rationing. The
environment itself applies no quota, no fine, and no restoration cost. All
regulation comes from the leader's mechanisms.

In [ ]:
ECOLOGY = {
    "r": 0.3,
    "K": 5_000,
    "p": 1.0,
    "B0": 4_000,
    "fish_init": 4_000,
    "sigma": 0.0,  # no process noise, so that the arithmetic below is exact
    "initial_stock_log_sigma": 0.0,
    "unregulated_f_multiplier": 2.0,
}
CAPACITY = float(ECOLOGY["K"])

bare_env = FisheryRegulatedEnv(
    world=None,
    mechanism_id=0,
    agents_cfg_dict={},
    leaders_cfg_dict={},
    schema=FisheryMetricSchema,
    ecology_cfg=ECOLOGY,
)
points = {name: bare_env.ecology[name] for name in ("B_msy", "MSY", "F_msy")}
print("reference points:", points)


def production(stock, r=0.3, p=1.0, capacity=CAPACITY):
    # Pella-Tomlinson production g(B), in biomass units per step.
    return (r / p) * stock * (1.0 - (stock / capacity) ** p)


for share in (0.1, 0.25, 0.5, 0.75, 1.0):
    print(f"B = {share:4.2f} K -> g(B) = {production(share * CAPACITY):7.1f}")
assert math.isclose(production(points["B_msy"]), points["MSY"])

The production is largest at $B_{\mathrm{MSY}}=K/2$ for the Schaefer case, where it
equals the maximum sustainable yield, and it vanishes at $B=K$. The next cell
advances the stock one step with the real transition `pella_tomlinson`, once with
requests that the stock can cover, and once with requests that exceed it, to show
the pro-rata rationing at work.

In [ ]:
def advance_one_step(requests):
    # State after one transition, given the catches that the fishers request.
    mdp = MDPState().add(bare_env.reset_fishery(MDPState()))
    mdp = mdp.add(MDPState(state={"catch_request": requests}))
    return bare_env.pella_tomlinson(mdp)


covered = advance_one_step({"a": 500.0, "b": 500.0})
rationed = advance_one_step({"a": 3_000.0, "b": 3_000.0})

# Requests of 1000 in total are delivered in full: 4000 + 240 - 1000.
print(
    "covered requests: next stock",
    covered.state["fish"][1],
    "catch",
    covered.state["usage"][1],
)
# Requests of 6000 exceed the stock of 4000: each fisher gets 2/3, so C = 4000.
print(
    "rationed requests: next stock",
    rationed.state["fish"][1],
    "catch",
    rationed.state["usage"][1],
)

assert math.isclose(covered.state["fish"][1], 4_000 + production(4_000.0) - 1_000)
assert math.isclose(rationed.state["fish"][1], production(4_000.0))
assert math.isclose(rationed.state["usage"][1], 4_000.0)

# 4. Quota mechanism

The quota is an **action-shaping mechanism**. It acts on the harvest request of
the fishers before the fishers' own mechanism turns that request into a catch.

## 4.1 Mathematical formulation

Let $b_t=B_t/K$ be the normalised biomass, and let $q\in[0,1]$ be the quota, the
regulator's action. Let $s(x)=1/(1+e^{-x})$ be the logistic function and $w_q$ the
quota transition width. The fraction of effort that the quota allows is

$$
\alpha_t
=
\frac{s\!\left(\frac{b_t-q}{w_q}\right)-s\!\left(\frac{-q}{w_q}\right)}
     {s\!\left(\frac{1-q}{w_q}\right)-s\!\left(\frac{-q}{w_q}\right)}.
$$

The two shifted terms normalise the logistic curve so that $\alpha_t=0$ when the
stock is empty and $\alpha_t=1$ when it is full. Far below the threshold $q$ the
allowance is close to $0$, far above it the allowance is close to $1$, and it
rises smoothly within a band of width $w_q$ around $q$.

A fisher whose raw output is $z$ requests the fraction $f=s(z/4)$. With a
second width $w_u$, the delivered fraction is the request minus a smooth excess
over the allowance:

$$
f^{*} = f-\operatorname{softplus}_0\!\left(f-\alpha_t;\,w_u\right),
\qquad
\operatorname{softplus}_0(x;w)=\max\left(0,\;w\left[\ln\left(1+e^{x/w}\right)-\ln 2\right]\right).
$$

The function $\operatorname{softplus}_0$ is a smooth version of $\max(x,0)$ that
is exactly zero for $x\le 0$, so $f^{*}\approx\min(f,\alpha_t)$. For a request far
above the allowance it stays below $\max(x,0)$ by $w_u\ln 2$, hence the delivered
fraction settles at $\alpha_t+w_u\ln 2$ and not exactly at $\alpha_t$. The
delivered fraction is also bounded to $[10^{-6},1-10^{-6}]$.

The regulator acts before the fishers, so at the moment the quota runs the request
has not been decoded yet and is still the raw output $z$. The quota therefore
returns a residual on $z$, which moves the request to the delivered fraction:

$$
\Delta z = 4\,\ln\frac{f^{*}}{1-f^{*}} - z.
$$

The residual is added to the fisher's raw action, and the fisher's own `Fishing`
mechanism then decodes $z+\Delta z$ as $s\big((z+\Delta z)/4\big)=f^{*}$. The
quota also records the allowance $\alpha_t$ in the state entry `allowed_frac`.
This smooth cap is a modelling choice of the fishery benchmark. It keeps the
delivered action differentiable in the quota and in the stock, and it is not taken
from a published model. The idea of an exogenous quota as a regulatory
institution for a common-pool resource is studied by Madani and Dinar (2013).

## 4.2 Quota configuration

The arguments of the `Quota` configuration built in section 2.1 have the
following meaning. The quota $q$ itself is not an argument: it is the regulator's
action, searched in $[0,1]$ by the outer optimizer through the declared
`action_space`, and clipped to that interval by `decode`. The argument `default`
is the quota in force before the first candidate reaches the environment, here
the value $0.56224$ of the canonical experiment.

The argument `acts_on=("fisherman", "harvest")` selects the targeted followers
and mechanism, and `obs_map={"resource_level": "fish"}` tells the mechanism to
read the resource from the `"fish"` entry of the state. The argument
`quota_transition_width` is $w_q$, the smoothness of the allowance around the
threshold, and `usage_transition_width` is $w_u$, the smoothness of the cap on the
request. Both are positive and have the defaults $0.03$ and $0.005$.

## 4.3 The quota at work

The next cell builds small shared states by hand, so that the mechanism can be
called on exact inputs. The helper `make_state` writes the fishers' raw policy
outputs and the regulator's action into the state, as the environment does. The
cell then applies the quota to a fisher that requests 90 percent of its maximal
catch, at several stock levels, and compares the result with the formulas of
section 4.1 implemented independently.

In [ ]:
QUOTA = float(np.float32(0.56224))  # the value the float32 action space stores
QUOTA_WIDTH = 0.03
USAGE_WIDTH = 0.005

fishing = FishingConfig(id="harvest", action_space=UNBOUNDED_BOX).build("fisherman:0")
restore = RestoreConfig(id="restore", action_space=UNBOUNDED_BOX).build("fisherman:0")


def raw_output(fraction):
    # Raw policy output z whose decoded fraction expit(z / 4) is `fraction`.
    z = ACTION_TEMPERATURE * math.log(fraction / (1.0 - fraction))
    return np.array([z], dtype=np.float32)


def make_state(
    *, stock, harvest, effort=None, regulator, capacity=CAPACITY, key="fish"
):
    # Shared state at step 0: the stock, the fishers' raw outputs, the regulator action.
    actions = {"regulator": regulator}
    for index, fraction in enumerate(harvest):
        actions[f"fisherman:{index}"] = {"harvest": raw_output(fraction)}
        if effort is not None:
            actions[f"fisherman:{index}"]["restore"] = raw_output(effort[index])
    return MDPState(params={"K": capacity}, state={key: stock}, actions=actions)


def allowed_fraction(level, quota, width=QUOTA_WIDTH):
    # The allowance alpha of section 4.1, written from the formula.
    low = expit(-quota / width)
    return (expit((level - quota) / width) - low) / (expit((1.0 - quota) / width) - low)


def delivered_fraction(requested, level, quota):
    # The delivered fraction f* of section 4.1, written from the formula.
    excess = requested - allowed_fraction(level, quota)
    softplus0 = max(
        0.0, USAGE_WIDTH * (np.logaddexp(0.0, excess / USAGE_WIDTH) - math.log(2.0))
    )
    return float(np.clip(requested - softplus0, 1e-6, 1.0 - 1e-6))


REQUESTED = 0.9
print(
    f"quota q = {QUOTA:.5f}; every fisher requests {REQUESTED:.0%} of its maximal catch"
)
print("stock level b | allowed alpha | delivered f* | cut applied to z")
for level in (0.20, 0.40, 0.50, QUOTA, 0.60, 0.70, 0.90):
    mdp = make_state(
        stock=level * CAPACITY,
        harvest=[REQUESTED],
        regulator={"quota": np.array([QUOTA], dtype=np.float32)},
    )
    residual = quota(mdp, np.array([QUOTA], dtype=np.float32))
    composed = mdp.add(residual)
    delivered = fishing.decode(None, composed.raw_actions["fisherman:0"]["harvest"][0])
    allowed = residual.state["allowed_frac"][0]
    cut = residual.actions["fisherman:0"]["harvest"][0][0]
    print(f"{level:13.5f} | {allowed:13.5f} | {delivered:12.5f} | {cut:+.4f}")

    assert abs(allowed - allowed_fraction(level, QUOTA)) < 1e-6
    assert abs(delivered - delivered_fraction(REQUESTED, level, QUOTA)) < 1e-5
    # Independent bracket: a request below the allowance passes untouched, and a
    # request above it is delivered between the allowance and allowance + w_u ln 2.
    if REQUESTED <= allowed:
        assert abs(delivered - REQUESTED) < 1e-6
    else:
        assert allowed - 1e-6 <= delivered <= allowed + USAGE_WIDTH * math.log(2) + 1e-6

At a stock far below the quota the allowance is close to zero and the quota cuts
the request almost to nothing. At a stock of 0.7 of the capacity the allowance is
above 0.9, so the request passes untouched and the residual on the action is
zero. At the threshold the allowance is approximately one half (exactly one half when $q=0.5$). The next figure draws
both behaviours by calling the mechanism on a grid of stock levels and of
requests.

In [ ]:
GRID = np.linspace(0.0, 1.0, 101)
COLORS = ["#7c3aed", "#ec4899", "#10b981"]


def call_quota(level, quota_value, requested):
    mdp = make_state(
        stock=level * CAPACITY,
        harvest=[requested],
        regulator={"quota": np.array([quota_value], dtype=np.float32)},
    )
    residual = quota(mdp, np.array([quota_value], dtype=np.float32))
    composed = mdp.add(residual)
    z_delivered = composed.raw_actions["fisherman:0"]["harvest"][0]
    return residual.state["allowed_frac"][0], fishing.decode(None, z_delivered)


plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.titleweight": "bold",
    }
)
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for color, quota_value in zip(COLORS, (0.3, QUOTA, 0.8)):
    allowed = [call_quota(level, quota_value, 0.5)[0] for level in GRID]
    left.plot(GRID, allowed, color=color, label=f"quota q = {quota_value:.2f}")
    left.axvline(quota_value, color=color, linestyle=":", alpha=0.6)
left.set_title("Allowed fraction of effort against the normalised stock", pad=12)
left.set_xlabel("Normalised stock b = B / K")
left.set_ylabel("Allowed fraction alpha [0-1]")
left.legend()

requests = np.linspace(0.02, 0.98, 49)
delivered = [call_quota(0.6, QUOTA, request)[1] for request in requests]
right.plot(requests, requests, color="#8b949e", linestyle="--", label="no quota")
right.plot(requests, delivered, color="#7c3aed", label="with the quota")
right.axhline(
    allowed_fraction(0.6, QUOTA),
    color="#ec4899",
    linestyle=":",
    label="allowance alpha",
)
right.set_title("Delivered against requested fraction at b = 0.6", pad=12)
right.set_xlabel("Requested harvest fraction f [0-1]")
right.set_ylabel("Delivered harvest fraction f* [0-1]")
right.legend()
plt.show()

**Read the figure.** In the left panel each curve is an S-shaped gate that opens
around its quota, and a lower quota opens the gate at a lower stock. In the right
panel the delivered fraction follows the diagonal while the request is below the
allowance, then flattens just above the dotted allowance line, which is the
smooth minimum of section 4.1.

The same algorithm regulates any resource that the environment normalises by a
capacity. The cell below applies it to a water reservoir by changing only the
names in `acts_on` and `obs_map` and the capacity in the parameters, and checks
that it gives the same allowance as the fishery at the same normalised level.

In [ ]:
water_quota = Quota(
    id="quota",
    action_space=UNIT_BOX,
    acts_on=("farmer", "withdrawal"),
    obs_map={"resource_level": "reservoir"},
).build("regulator")

reservoir = MDPState(
    params={"K": 200.0},
    state={"reservoir": 120.0},  # 0.6 of the capacity
    actions={
        "regulator": {"quota": np.array([QUOTA], dtype=np.float32)},
        "farmer:0": {"withdrawal": raw_output(0.9)},
    },
)
water_residual = water_quota(reservoir, np.array([QUOTA], dtype=np.float32))
water_allowed = water_residual.state["allowed_frac"][0]
fish_allowed = call_quota(0.6, QUOTA, 0.9)[0]
print(f"reservoir allowance {water_allowed:.5f}; fishery allowance {fish_allowed:.5f}")
assert abs(water_allowed - fish_allowed) < 1e-6

# 5. Learned incentive and restoration subsidy

The subsidy is a **reward-shaping mechanism**. The benchmark determines the
ecological effect of restoration, and the mechanism determines the incentive
attached to it.

## 5.1 General learned incentive

A learned incentive designer, as in the adaptive incentive design of Yang et al.
(2022), can be written as a function $u_t=\mu_\eta(s_t,a_t)$ that is itself
trained by meta-gradients, and the reward of agent $i$ becomes

$$
R_\eta^{(i)}(s_t,a_t)
=
R^{(i)}\left(s_t,a_t,\mu_\eta(s_t,a_t)\right).
$$

The subsidy of this codebase is much simpler. It is not a neural incentive
network: the regulator searches a single scalar rate with an evolution strategy,
and the mechanism pays that rate for the effort that the followers choose.

## 5.2 Fish habitat restoration incentive

Let $e_{i,t}=s(z^{e}_{i,t}/4)$ be the restoration effort of fisher $i$, let
$\kappa_t\in[0,1]$ be the regulator's action, and let $c\in[0,1]$ be a fixed
quadratic cost. With the constant `MAX_SUBSIDY` $=0.5$, the payment per unit of effort is
$\pi_t=\mathrm{MAX\_SUBSIDY}\cdot\kappa_t$, and the mechanism adds to the
reward of every targeted fisher

$$
\Delta r_{i,t}
=
\pi_t\,e_{i,t}-c\,e_{i,t}^{2}.
$$

The environment's own reward, the harvest fraction, does not contain the stock,
so within the environment a fisher has no direct reason to restore. Restoration
pays back only indirectly, because a larger stock raises the allowance of a quota
and lowers the deduction of a threshold penalty when the regulator holds them. The payment is linear in the
effort and the cost is convex, hence the net payment is a concave parabola in
$e$. It peaks at $e=\pi_t/(2c)$ when that value lies in $(0,1)$, and the
quadratic cost is what discourages saturating the effort at its maximum. This
functional form is a modelling choice of the fishery benchmark and is not taken
from a published model.

## 5.3 Subsidy configuration

The rate $\kappa_t$ is the regulator's action and is searched in $[0,1]$. The cost
$c$ is the fixed argument `cost` of the configuration, and the mechanism rejects a
value outside $[0,1]$. The argument `acts_on=("fisherman", "restore")` selects the
follower mechanism whose action is the subsidised effort. The cell below builds
the mechanism and calls it on a range of efforts, for a rate of one and for a
smaller rate, and checks the payment against the formula of section 5.2.

In [ ]:
EFFORT_COST = 0.25

subsidy_cfg = Subsidy(
    id="subsidy",
    action_space=UNIT_BOX,
    acts_on=("fisherman", "restore"),
    cost=EFFORT_COST,
)
subsidy = subsidy_cfg.build("regulator")


def subsidy_payment(effort, rate):
    # Reward added to one fisher that makes `effort` under the regulator rate.
    mdp = make_state(
        stock=0.6 * CAPACITY,
        harvest=[0.5],
        effort=[effort],
        regulator={"subsidy": np.array([rate], dtype=np.float32)},
    )
    residual = subsidy(mdp, np.array([rate], dtype=np.float32))
    return residual.rewards["fisherman:0"][0]


print("effort e | payment at rate 1.0 | payment at rate 0.4")
for effort in (0.05, 0.25, 0.50, 0.75, 0.95):
    full, small = subsidy_payment(effort, 1.0), subsidy_payment(effort, 0.4)
    print(f"{effort:8.2f} | {full:19.5f} | {small:19.5f}")
    # The scale 0.5 is the value of MAX_SUBSIDY that section 5.2 states.
    assert abs(full - (1.0 * 0.5 * effort - EFFORT_COST * effort**2)) < 1e-6
    assert abs(small - (0.4 * 0.5 * effort - EFFORT_COST * effort**2)) < 1e-6
assert MAX_SUBSIDY == 0.5

efforts = np.linspace(0.01, 0.99, 99)
best = efforts[np.argmax([subsidy_payment(effort, 0.4) for effort in efforts])]
print(f"most profitable effort at rate 0.4: {best:.2f} (theory: pi/(2c) = 0.40)")
assert abs(best - 0.4) < 0.011

## 5.4 Ecology versus incentive

The ecological effect of restoration belongs to the transition, and the incentive
belongs to the reward. They are two mechanisms in two different places. The
fisher's own `Restore` mechanism turns the effort into biomass,

$$
R_t=\frac{\rho K}{N}\sum_i e_{i,t},
$$

where $\rho$ is the `restoration_effectiveness` of the ecology, and the transition
adds $R_t$ to the stock in the equation of section 3. The regulator's `Subsidy`
touches only the rewards. The cell below checks both halves: the restored biomass
of two fishers, and the fact that the subsidy residual leaves the state alone.

In [ ]:
restoring = MDPState(
    params={"restoration_effectiveness": 0.02, "K": CAPACITY},
    aids={"fisherman:0", "fisherman:1"},
)
per_fisher = restore.apply(restoring, 0.5).state["restoration"][0]
print("biomass restored by one fisher at effort 0.5:", per_fisher)
assert math.isclose(per_fisher, 0.02 * CAPACITY * 0.5 / 2)
print("two fishers together restore:", 2 * per_fisher, "= rho * K * mean effort")

subsidy_mdp = make_state(
    stock=0.6 * CAPACITY,
    harvest=[0.5, 0.5],
    effort=[0.8, 0.2],
    regulator={"subsidy": np.array([0.6], dtype=np.float32)},
)
residual = subsidy(subsidy_mdp, np.array([0.6], dtype=np.float32))
print(
    "fields the subsidy residual holds:",
    {
        "rewards": sorted(residual.rewards.data),
        "state": sorted(residual.state.data),
        "actions": sorted(residual.actions.data),
    },
)
assert residual.state.data == {} and residual.actions.data == {}

# 6. Threshold penalty

The penalty is a second **reward-shaping mechanism**. Where the subsidy pays for
an action that the regulator wants, the penalty sanctions every fisher while the
resource is depleted, whatever that fisher does.

## 6.1 Mathematical formulation

Let $b_t=B_t/K$ be the normalised biomass, $\tau\in[0,1]$ the threshold,
$\lambda\ge 0$ the penalty amount and $w_p>0$ the transition width. The mechanism
adds the same deduction to the reward of every targeted fisher $i$:

$$
\Delta r_{i,t}
=
-\frac{\lambda}{1+\exp\!\left(\frac{b_t-\tau}{w_p}\right)}.
$$

The deduction approaches $\lambda$ when the stock is far below $\tau$, it
approaches $0$ when the stock is far above it, and it equals exactly $\lambda/2$
at $b_t=\tau$. The logistic transition keeps the reward continuous in the stock,
which is a smoothing choice of this codebase and not a published model. The
deduction does not depend on the fisher's own action, so the penalty is a
collective sanction tied to the state of the resource. To act on an individual
action, use the quota of section 4.

Only the agent type of `acts_on` is used by the penalty: every agent whose
identifier starts with `"<agent>:"` is sanctioned, whatever the mechanism named
in the second element. The pair `("fisherman", "harvest")` therefore does not
restrict the penalty to the harvest.

The penalty is a fixed regulatory rule. Its three numbers $\tau$, $\lambda$ and
$w_p$ are arguments of the configuration, with the defaults $0.20$, $0.10$ and
$0.03$, and the regulator searches nothing for it: its `action_space` is empty,
so it adds no dimension to the outer search.

In [ ]:
penalty_cfg = ThresholdPenalty(
    id="penalty",
    acts_on=("fisherman", "harvest"),
    obs_map={"resource_level": "fish"},
    threshold=0.20,
    penalty_amount=0.10,
    transition_width=0.03,
)
penalty = penalty_cfg.build("regulator")
print(
    "action space of the penalty (empty, nothing to search):", penalty_cfg.action_space
)

NO_ACTION = np.empty(0, dtype=np.float32)
print("level b | deduction by formula | deduction of each of the three fishers")
for level in (0.05, 0.12, 0.20, 0.28, 0.40, 0.80):
    mdp = make_state(
        stock=level * CAPACITY,
        harvest=[0.2, 0.9, 0.5],  # three fishers that behave very differently
        regulator={"penalty": NO_ACTION},
    )
    residual = penalty(mdp, NO_ACTION)
    deductions = [residual.rewards[f"fisherman:{i}"][0] for i in range(3)]
    expected = -0.10 / (1.0 + math.exp((level - 0.20) / 0.03))
    print(f"{level:7.2f} | {expected:20.5f} | {np.round(deductions, 5)}")
    assert all(abs(value - expected) < 1e-9 for value in deductions)

assert math.isclose(penalty.penalty(0.20), 0.05)  # half of the amount at the threshold
print("the residual holds only rewards:", sorted(residual.rewards.data))
assert residual.state.data == {} and residual.actions.data == {}

The table confirms the three properties. The deduction is the same for the three
fishers, although one requests 20 percent of its maximal catch and another 90
percent. It is half of the amount at the threshold, and it fades above it. The
residual holds only the reward field, so the penalty cannot change the stock or
the actions.

# 7. Social influence

Social influence is an **observation-shaping mechanism**. The quota, the subsidy
and the penalty change what the fishers do or what they earn. This one changes
only what they see.

## 7.1 Mathematical formulation

Consider the $N$ targeted fishers whose targeted mechanism has a $d$-dimensional
action, and let $a_{j,t-1}$ be the action that fisher $j$ **delivered** at step
$t-1$. With $k$ the configured `obs_offset`, the mechanism adds to the observation
of fisher $i$ at step $t$

$$
o_{i,t}\left[k : k+(N-1)\,d\right]
\;\leftarrow\;
\left[\,a_{j,t-1}\;:\; j\neq i\,\right],
$$

with the peers ordered by identifier. The contribution is summed with the
observation that each fisher builds on its own, so the fishers must leave those
entries at zero, and the fishery observation `[b, 0, usage/K, 0, 0]` does leave
entries $1$, $3$ and $4$ free. Nothing is exposed at reset, because no action
exists yet.

The exposed value is the delivered action, after the regulator's other mechanisms
and after the fisher's own decoding, and not the raw policy output. With a quota
in force a fisher therefore sees what its peers were allowed to request. The
mechanism has no searched parameter, so its action space is empty, and it
contributes nothing to the transition: only `observe` returns a residual. To
expose two mechanisms, declare one instance per mechanism with disjoint entries.

Jaques et al. (2019) use the same name for a richer idea: an intrinsic reward
that pays an agent for the causal influence of its action on the policies of the
other agents, measured with a Kullback-Leibler divergence. That reward is **not**
implemented here. The argument `influence_weight` is reserved for it and has no
effect, so this mechanism should be read as making behaviour public, in the
spirit of social learning, and not as the algorithm of the paper.

## 7.2 Social influence at work

The cell below gives three fishers the five-entry fishery observation and calls
`observe` at step 1, once the first actions exist. Each fisher receives the
actions of the other two in entries 3 and 4. A fourth fisher would need a third
entry, which does not exist in a five-entry observation, so the mechanism refuses
with an explicit error instead of writing outside the vector.

In [ ]:
social = SocialInfluence(
    id="social_harvest", acts_on=("fisherman", "harvest"), obs_offset=3
).build("regulator")

delivered_harvest = {"fisherman:0": 0.25, "fisherman:1": 0.50, "fisherman:2": 0.75}
mdp = MDPState(
    actions={aid: {"harvest": value} for aid, value in delivered_harvest.items()},
    obs={aid: np.zeros(5) for aid in delivered_harvest},
)

contribution = social.observe(dataclasses.replace(mdp, t=1))
for aid in delivered_harvest:
    print(aid, "receives", contribution.obs[aid][0])

# fisherman:0 sees the actions of fisherman:1 and fisherman:2, in that order.
assert contribution.obs["fisherman:0"][0].tolist() == [0, 0, 0, 0.5, 0.75]
assert contribution.obs["fisherman:2"][0].tolist() == [0, 0, 0, 0.25, 0.5]
assert social.observe(mdp).obs.data == {}  # nothing at reset (t = 0)
transition_residual = social.apply(mdp, NO_ACTION)  # nothing in the transition
assert all(
    field.data == {}
    for field in (
        transition_residual.state,
        transition_residual.actions,
        transition_residual.rewards,
        transition_residual.obs,
    )
)

crowded = MDPState(
    actions={f"fisherman:{i}": {"harvest": 0.1 * i} for i in range(4)},
    obs={f"fisherman:{i}": np.zeros(5) for i in range(4)},
)
try:
    social.observe(dataclasses.replace(crowded, t=1))
except ValueError as error:
    print("four fishers do not fit:", error)
else:
    raise AssertionError("the mechanism must refuse four fishers")

# 8. Composition: several mechanisms in one regulator

A regulator holds a tuple of mechanisms, and the tuple is the only composition
rule. There is no chaining or parallel combinator: each mechanism returns a
residual, and the residuals are added. This section states the semantics, checks
them on a bare state, and then checks them again through the real environment.

## 8.1 What adding residuals means

When an agent acts, each of its mechanisms is applied to the **same** input
state, and the residuals are summed with `MDPState.add`. Two consequences follow.
First, the order of the mechanisms does not matter. Second, two residuals on the
same channel add up and do not feed one another. Two quotas on the same fishers
do not give the smaller of the two allowances: each one computes its own cut of
the raw output $z$ from the uncut request, and both cuts are applied, so the
fisher is cut twice.

$$
z^{*}=z+\Delta z_{1}+\Delta z_{2},
\qquad
\Delta z_{k}=4\ln\frac{f_k^{*}}{1-f_k^{*}}-z.
$$

The cell below applies two quotas, in both orders, to a fisher that requests 90
percent at a stock of 0.6 of the capacity, and compares the delivered fraction
with that of a single quota.

In [ ]:
second_quota = Quota(
    id="quota_b",
    action_space=UNIT_BOX,
    acts_on=("fisherman", "harvest"),
    obs_map={"resource_level": "fish"},
).build("regulator")

QUOTA_ACTION = np.array([0.7], dtype=np.float32)


def deliver(order):
    # Fraction delivered when the quotas of `order` act on the same input state.
    mdp = make_state(
        stock=0.6 * CAPACITY,
        harvest=[REQUESTED],
        regulator={"quota": QUOTA_ACTION, "quota_b": QUOTA_ACTION},
    )
    residuals = {
        "quota": quota(mdp, QUOTA_ACTION),
        "quota_b": second_quota(mdp, QUOTA_ACTION),
    }
    for name in order:
        mdp = mdp.add(residuals[name])
    return fishing.decode(None, mdp.raw_actions["fisherman:0"]["harvest"][0])


single = delivered_fraction(REQUESTED, 0.6, 0.7)
z_raw = ACTION_TEMPERATURE * math.log(REQUESTED / (1 - REQUESTED))
cut = ACTION_TEMPERATURE * math.log(single / (1 - single)) - z_raw
twice = float(expit((z_raw + 2 * cut) / ACTION_TEMPERATURE))

print(f"requested {REQUESTED:.4f} | one quota {single:.4f} | two quotas {twice:.4f}")
print("computed with the mechanisms:", deliver(("quota", "quota_b")))
assert abs(deliver(("quota", "quota_b")) - deliver(("quota_b", "quota"))) < 1e-9
assert abs(deliver(("quota", "quota_b")) - twice) < 1e-5
assert twice < single < REQUESTED

The order does not change the result, and two identical quotas deliver less than
one, so a regulator should hold one quota per targeted channel. Different
mechanisms on different channels, such as a quota on the harvest and a subsidy on
the reward, do not interact at all, which is the case that the next cells check.

## 8.2 Stepping the real fishery

The environment of the benchmark runs a regulator's mechanisms inside the step of
`MultiAgentEnv`. The leaders act before the followers. At each reset the
environment asks the `World` actor for the candidate that the outer optimizer has
published, and until a candidate exists each leader plays the `default` action of
its mechanisms. A mechanism without a default holds no action and contributes
nothing.

The cells below drive the real `FisheryRegulatedEnv` behind the RLlib adapter,
with two fishers and 20 steps. Only the `World` actor is replaced, by a script of
candidates, so that no Ray process is needed. This is what the integration tests
of the repository do. The adapter keeps the shared state in `_mdp`, which the
cells read only to inspect what was delivered.

In [ ]:
SOCIETY_ECOLOGY = {
    "r": 0.3,
    "K": CAPACITY,
    "p": 1.0,
    "B0": 4_000,
    "fish_init": 4_000,
    "sigma": 0.02,
    "initial_stock_log_sigma": 0.05,
    "unregulated_f_multiplier": 2.0,
}
REGULATOR_ID = "fisheries_regulator"
WORK_DIR = tempfile.TemporaryDirectory()


class ScriptedWorld:
    # Stand-in for the World actor: each fetch returns the next scripted answer.

    def __init__(self, answers):
        self.answers = list(answers)
        self.calls = 0
        self.get_mechanism_by_id = SimpleNamespace(remote=self._fetch)

    def _fetch(self, **kwargs):
        self.calls += 1
        return self.answers.pop(0) if self.answers else None


def candidate(mechanism):
    # The candidate that the outer optimizer publishes: one array per mechanism.
    return MechanismContext(
        index=0,
        env_id=None,
        seed=0,
        status=MechanismStatus.train,
        mechanism=mechanism,
        metrics=None,
    )


def build_fishery(*regulator_mechanisms, answers, num_agents=2, horizon=20):
    unbounded = spaces.Box(-np.inf, np.inf, (5,), np.float32)
    fisherman = FishermanConfig(
        id="fisherman",
        policy_id="fisher_policy",
        mechanisms=(
            FishingConfig(id="harvest", action_space=UNBOUNDED_BOX),
            RestoreConfig(id="restore", action_space=UNBOUNDED_BOX),
        ),
        observation_space=unbounded,
    )
    followers = {
        f"fisherman:{i}": dataclasses.replace(fisherman, id=f"fisherman:{i}")
        for i in range(num_agents)
    }
    regulator = AgentConfig(
        id=REGULATOR_ID, policy_id="quota_policy", mechanisms=regulator_mechanisms
    )
    world = ScriptedWorld(answers)
    env = FisheryRegulatedEnv(
        world=world,
        opt_id="inner",
        env_name="fishery_tutorial",
        horizon=horizon,
        agents_cfg_dict=followers,
        leaders_cfg_dict={REGULATOR_ID: regulator},
        mechanism_id=0,
        seed=0,
        policy_seed=0,
        mode="train",
        reporter_cfg=CSVConfig(project="tutorial", output_dir=WORK_DIR.name),
        queries=(),
        schema=FisheryMetricSchema,
        ecology_cfg=SOCIETY_ECOLOGY,
    )
    return SimpleNamespace(adapter=RLlibMultiAgentEnvAdapter(env), world=world)


def run_episodes(fishery, *, episodes=1, steps=20, seed=0, z_scale=2.0):
    # Step the fishery with seeded policy outputs and record what happens.
    adapter = fishery.adapter
    rng = np.random.default_rng(seed)
    finished = []
    # The environment fetches the candidate with ray.get; the script needs no Ray.
    # The patch replaces `get` on the ray module itself, so it is global for the
    # duration of the block, as in the integration conftest of the repository.
    with patch.object(marl_regulated.ray, "get", lambda ref, *a, **k: ref):
        for _ in range(episodes):
            reset_obs, _ = adapter.reset()
            records = []
            for _ in range(steps):
                # No public accessor exposes the shared state, hence the private
                # attribute; it is read only to inspect the episode.
                state = adapter._mdp
                t = state.t
                level = float(state.state["fish"][t]) / CAPACITY
                z = {
                    aid: {
                        "harvest": (z_scale * rng.normal(size=1)).astype(np.float32),
                        "restore": (z_scale * rng.normal(size=1)).astype(np.float32),
                    }
                    for aid in adapter.possible_agents
                }
                obs, rewards, *_ = adapter.step(z)
                delivered = {
                    aid: {
                        name: float(adapter._mdp.actions[aid][name][t])
                        for name in ("harvest", "restore")
                    }
                    for aid in adapter.possible_agents
                }
                records.append(
                    {
                        "t": t,
                        "level": level,
                        "z": {
                            aid: {k: float(v[0]) for k, v in outputs.items()}
                            for aid, outputs in z.items()
                        },
                        "delivered": delivered,
                        "obs": obs,
                        "rewards": rewards,
                    }
                )
            finished.append({"reset_obs": reset_obs, "steps": records})
    return finished

The first run gives the regulator a quota, a subsidy and a penalty, and publishes
one candidate for them. The expected reward of each fisher at each step is
written from the three formulas above, with the stock read before the step, and
not from the mechanisms. It is the delivered harvest fraction of section 4, plus
the subsidy payment of section 5, plus the penalty deduction of section 6.

In [ ]:
QUOTA_VALUE = float(np.float32(0.78))
SUBSIDY_RATE = float(np.float32(0.6))
SUBSIDY_COST = 0.5  # distinct from EFFORT_COST of section 5
PENALTY_TAU, PENALTY_LAMBDA, PENALTY_WIDTH = 0.77, 0.1, 0.03

three_mechanisms = (
    Quota(
        id="quota",
        action_space=UNIT_BOX,
        acts_on=("fisherman", "harvest"),
        obs_map={"resource_level": "fish"},
    ),
    Subsidy(
        id="subsidy",
        action_space=UNIT_BOX,
        acts_on=("fisherman", "restore"),
        cost=SUBSIDY_COST,
    ),
    ThresholdPenalty(
        id="penalty",
        acts_on=("fisherman", "harvest"),
        obs_map={"resource_level": "fish"},
        threshold=PENALTY_TAU,
        penalty_amount=PENALTY_LAMBDA,
        transition_width=PENALTY_WIDTH,
    ),
)
published = candidate(
    {
        "quota": np.array([QUOTA_VALUE], dtype=np.float32),
        "subsidy": np.array([SUBSIDY_RATE], dtype=np.float32),
        "penalty": NO_ACTION,
    }
)

fishery = build_fishery(*three_mechanisms, answers=[published])
(episode,) = run_episodes(fishery, episodes=1, steps=20, seed=0)


def expected_reward(step, aid):
    level = step["level"]
    requested = float(expit(step["z"][aid]["harvest"] / ACTION_TEMPERATURE))
    effort = float(expit(step["z"][aid]["restore"] / ACTION_TEMPERATURE))
    harvest = delivered_fraction(requested, level, QUOTA_VALUE)
    subsidy_term = SUBSIDY_RATE * MAX_SUBSIDY * effort - SUBSIDY_COST * effort**2
    penalty_term = -PENALTY_LAMBDA / (
        1 + math.exp((level - PENALTY_TAU) / PENALTY_WIDTH)
    )
    return requested, harvest, subsidy_term, penalty_term


print("step | stock b | requested f | delivered f* | subsidy | penalty | reward")
cut_by_quota = []
for step in episode["steps"]:
    for aid in ("fisherman:0", "fisherman:1"):
        requested, harvest, subsidy_term, penalty_term = expected_reward(step, aid)
        expected = harvest + subsidy_term + penalty_term
        assert abs(step["rewards"][aid] - expected) < 1e-5
        assert abs(step["delivered"][aid]["harvest"] - harvest) < 1e-5
        cut_by_quota.append(requested - harvest)
        if aid == "fisherman:0" and step["t"] < 6:
            print(
                f"{step['t']:4d} | {step['level']:7.4f} | {requested:11.4f} | "
                + f"{harvest:12.4f} | {subsidy_term:+7.4f} | {penalty_term:+7.4f} | "
                + f"{step['rewards'][aid]:+.4f}"
            )

print("largest cut applied by the quota:", round(max(cut_by_quota), 4))
assert max(cut_by_quota) > 0.05  # the quota binds on some requests
assert min(cut_by_quota) < 1e-3  # and lets others through untouched
print("the regulator was asked for its candidate", fishery.world.calls, "time(s)")

Every reward of the 40 records matches the sum of the three formulas to within
single-precision rounding, so the three residuals of one regulator compose
exactly as section 8.1 states, and the quota both binds on some requests and
leaves others untouched. The reward the fisher is paid is the harvest fraction
*after* the quota, which is also the quantity that the quota lowers.

The second run adds two social-influence mechanisms, one for the harvest in entry
3 and one for the restoration in entry 4. The peer's entries must hold the action
that the peer *delivered* at the previous step, which is the quota-limited one,
and the entries must be zero at reset.

In [ ]:
social_mechanisms = (
    SocialInfluence(
        id="social_harvest", acts_on=("fisherman", "harvest"), obs_offset=3
    ),
    SocialInfluence(
        id="social_restore", acts_on=("fisherman", "restore"), obs_offset=4
    ),
)
social_candidate = candidate(
    {
        "quota": np.array([QUOTA_VALUE], dtype=np.float32),
        "subsidy": np.array([SUBSIDY_RATE], dtype=np.float32),
        "penalty": NO_ACTION,
        "social_harvest": NO_ACTION,
        "social_restore": NO_ACTION,
    }
)
fishery = build_fishery(
    *three_mechanisms, *social_mechanisms, answers=[social_candidate]
)
(episode,) = run_episodes(fishery, episodes=1, steps=20, seed=1)

for obs in episode["reset_obs"].values():
    assert obs[3] == 0.0 and obs[4] == 0.0  # nothing is exposed at reset

shown = 0
for step in episode["steps"]:
    for aid, obs in step["obs"].items():
        (peer,) = [other for other in step["obs"] if other != aid]
        assert obs.shape == (5,) and obs.dtype == np.float32
        assert obs[1] == 0.0  # an entry that no mechanism owns stays at zero
        # The peer's delivered actions at this same step (stored at step t).
        assert obs[3] == np.float32(step["delivered"][peer]["harvest"])
        assert obs[4] == np.float32(step["delivered"][peer]["restore"])
        # The delivered values, from the formulas of sections 4 and 5 and the
        # stock of this step, not from the stored actions.
        peer_z = step["z"][peer]
        peer_f = float(expit(peer_z["harvest"] / ACTION_TEMPERATURE))
        assert (
            abs(obs[3] - delivered_fraction(peer_f, step["level"], QUOTA_VALUE)) < 1e-5
        )
        assert abs(obs[4] - float(expit(peer_z["restore"] / ACTION_TEMPERATURE))) < 1e-5
        if aid == "fisherman:0" and shown < 4:
            shown += 1
            peer_request = float(expit(step["z"][peer]["harvest"] / ACTION_TEMPERATURE))
            print(
                f"t={step['t']}: fisherman:0 sees fisherman:1 deliver "
                + f"harvest {obs[3]:.4f} (it requested {peer_request:.4f}) "
                + f"and restore {obs[4]:.4f}"
            )

The observation is the sum of two things: the entries that the fisher builds
itself (stock in entry 0, usage in entry 2) and the entries that the leader's
mechanisms add. The assertions show that each mechanism writes only its own entry
and that the peers' actions are the delivered ones.

The third run shows the role of `default`. The regulator holds only a quota, with
a default of $0.56224$ that the first episode plays because no candidate exists
yet. The second episode fetches a candidate of $0.85$, which is above the stock
of about $0.8$ and therefore cuts the requests hard. The reward of each episode
is checked against the quota that was in force.

In [ ]:
default_quota = Quota(
    id="quota",
    action_space=UNIT_BOX,
    acts_on=("fisherman", "harvest"),
    obs_map={"resource_level": "fish"},
    default=np.asarray(0.56224, dtype=np.float32),
)
strict = candidate({"quota": np.array([0.85], dtype=np.float32)})
fishery = build_fishery(default_quota, answers=[None, strict], horizon=5)
first, second = run_episodes(fishery, episodes=2, steps=5, seed=2)

for name, run, quota_in_force in (
    ("default", first, float(np.float32(0.56224))),
    ("candidate", second, float(np.float32(0.85))),
):
    gaps = []
    for step in run["steps"]:
        for aid, reward in step["rewards"].items():
            requested = float(expit(step["z"][aid]["harvest"] / ACTION_TEMPERATURE))
            expected = delivered_fraction(requested, step["level"], quota_in_force)
            assert abs(reward - expected) < 1e-5
            gaps.append(requested - reward)
    print(f"quota {quota_in_force:.5f} ({name}): mean cut {np.mean(gaps):.4f}")

assert fishery.world.calls == 2  # one fetch per reset until a candidate arrives

The default quota lies far below the stock, so its allowance is close to one and
it cuts almost nothing, while the candidate above the stock cuts a large part of
every request. The environment asked the `World` for a candidate at each of its
two resets, and played the default only for the episode that ran before the
candidate existed.

# 9. How the outer optimizer sees several mechanisms

The outer optimizer does not know about quotas or subsidies. It sees the regulator
as a gymnasium space: the dictionary of the action spaces of all its mechanisms,
keyed by mechanism id, which gymnasium sorts alphabetically. The optimizer is an
evolution strategy in the family of Salimans et al. (2017), which perturbs a mean
vector and moves it towards the better candidates. It searches the flattened
vector of that space, with every coordinate bounded to $[0,1]$, and each
candidate vector is unflattened into one array per mechanism before it is
published. Its sigma adaptation and logit parametrisation are design choices of
this implementation and are not part of the cited method.

A mechanism with an empty action space contributes no coordinate. The penalty and
the two social-influence mechanisms are therefore carried along in every
candidate, with empty arrays, without enlarging the search. The search mean starts
at $0.5$ for every coordinate: the `default` of a mechanism is played before the
first candidate, but it is not the starting point of the search, which the
`initial_mean` option of `ESConfig.training` sets explicitly. The optimizer can
be built without Ray, so the cell below inspects it directly.

In [ ]:
full_stack = (
    Quota(
        id="quota",
        action_space=UNIT_BOX,
        acts_on=("fisherman", "harvest"),
        obs_map={"resource_level": "fish"},
        default=np.asarray(0.56224, dtype=np.float32),
    ),
    Subsidy(
        id="subsidy",
        action_space=UNIT_BOX,
        acts_on=("fisherman", "restore"),
        cost=0.25,
        default=np.asarray(0.2, dtype=np.float32),
    ),
    ThresholdPenalty(
        id="penalty",
        acts_on=("fisherman", "harvest"),
        obs_map={"resource_level": "fish"},
    ),
    *social_mechanisms,
)
regulator_cfg = AgentConfig(
    id=REGULATOR_ID, policy_id="quota_policy", mechanisms=full_stack
)
es_cfg = ESConfig().training(sigma=0.15, mean_lr=0.10, episodes=2).agents(regulator_cfg)
es = ESOptimizer(es_cfg.debugging(seed=42))

print("mechanism ids in the space:", list(es.action_space.spaces))
print("searched dimension:", es.dimension, "| parameter names:", es.parameter_names)
print("initial mean of the search:", es.mean)
assert es.dimension == 2 and es.parameter_names == ["quota", "subsidy"]
assert np.allclose(es.mean, 0.5)

# The optimizer unflattens each candidate with gymnasium's `unflatten`.
vector = np.array([0.7, 0.1], dtype=np.float32)
published_action = unflatten(es.action_space, vector)
for name, value in published_action.items():
    print(f"{name:15s} -> {value}")
assert list(es.action_space.spaces) == sorted(m.id for m in full_stack)

# `initial_mean` starts the search at chosen values, for example the defaults.
seeded = ESOptimizer(
    ESConfig()
    .training(sigma=0.15, mean_lr=0.10, episodes=2, initial_mean=[0.56224, 0.2])
    .agents(regulator_cfg)
    .debugging(seed=42)
)
print("search started at the defaults:", seeded.mean)

fixed_only = ESOptimizer(
    ESConfig()
    .training(episodes=2)
    .agents(
        AgentConfig(
            id=REGULATOR_ID, policy_id="quota_policy", mechanisms=(penalty_cfg,)
        )
    )
    .debugging(seed=42)
)
print("a regulator with only the penalty has dimension", fixed_only.dimension)
assert fixed_only.fixed_mode

The two searched coordinates are the quota and the subsidy rate, in alphabetical
order. A regulator that holds only fixed rules, such as the penalty, has
dimension zero, and the optimizer then runs in a fixed mode in which the same
candidate is evaluated at every generation.

# 10. What the regulator is optimized for

The outer optimizer maximises a fitness, and the fitness is not the fishers'
reward. For each candidate, `FisheryRegulatorEnv` reads the series that the inner
environment logged during the evaluation episodes, keeps the **last 50 steps** of
each episode, and computes

$$
\text{fitness}
=
\underbrace{\frac{\overline{H}}{\mathrm{MSY}}}_{\text{harvest score}}
+\;w_s\,\overline{b},
$$

where $\overline{H}$ is the mean *realised* harvest of the tail, $\mathrm{MSY}$
the maximum sustainable yield, $\overline{b}$ the mean normalised biomass of the
tail, and $w_s$ the `sustainability_weight`. The statistics are averaged over the
episodes and seeds of the candidate. The window measures the steady state that
the episode reaches, so that the transient of the first steps does not decide
which mechanism wins. The weight $w_s$ and the window length are choices of the
benchmark and are not taken from the literature.

Two quantities that look alike differ here. The fisher is paid the harvest
fraction after the quota and **before** the pro-rata rationing of section 3, so
the reward can exceed what the fisher actually catches when the stock is short.
The fitness uses the realised harvest, which is the rationed catch. The regulator
therefore optimises a different quantity from the one that the fishers learn from.

The cell below scores one candidate with the real `reward` method. The stand-in
world and metrics follow the example in the class docstring, with an episode of
120 steps whose first 70 steps are a collapse and whose last 50 are healthy. The
fitness ignores the collapse and equals $0.5+2\times 0.5$.

In [ ]:
appended = []
stand_in_world = SimpleNamespace(
    append_context=SimpleNamespace(remote=lambda context: appended.append(context))
)
regulator_env = FisheryRegulatorEnv(
    world=stand_in_world,
    optimizer=None,
    horizon=1,
    agents_cfgs={},
    seeds=[0],
    ecology_cfg={"sustainability_weight": 2.0},
)
print("steps used by the fitness:", regulator_env.fitness_tail_steps)

MSY_VALUE = 75.0
logged = SimpleNamespace(
    reward_series=[[0.1] * 70 + [0.5] * 50],
    fish_norm_next_series=[[0.1] * 70 + [0.5] * 50],
    H_realized_series=[[0.0] * 70 + [37.5] * 50],
    MSY=[MSY_VALUE],
)
seed_metrics = SimpleNamespace(by_episode={"0": logged})
rollout = SimpleNamespace(
    by_mechanism={"0": SimpleNamespace(by_seed={"0": seed_metrics})}
)
metrics = SimpleNamespace(eval=SimpleNamespace(rollout=rollout))

with patch.object(ray, "get", lambda ref: ref):
    fitness = regulator_env.reward(metrics)

harvest_score = 37.5 / MSY_VALUE
print("fitness of the candidate:", fitness, "| by hand:", harvest_score + 2.0 * 0.5)
assert math.isclose(fitness[0], harvest_score + 2.0 * 0.5, rel_tol=1e-6)
published_fitness = appended[0].payload.metrics
print("published to the world:", published_fitness.objective_score)

# 11. The full bilevel configuration

The two levels are assembled with the fluent `BilevelConfig` builder, which has
one call per concern: the `World` actor, the reporter, the Ray runtime, the
regulator (outer level) and the society (inner level). The function below
reproduces `examples/bilevel_fishery/config.yaml`, with four differences. The
reporter writes CSV files into a temporary directory instead of calling Weights
and Biases with its options. Ray logs at the `INFO` level instead of `ERROR`, so
that its messages stay visible. The `runtime_env` block that excludes working
directories from the Ray upload is dropped, because the notebook does not ship
the repository to workers. Finally, the sizes are arguments, so that the same
function builds the canonical experiment and a tiny one.

Note what the regulator is given. Its environment receives only the objective
settings, `sustainability_weight` and `sustainability_threshold`, and no carrying
capacity: the objective reads the biomass that the inner environment has already
normalised by its own `K`, so the capacity lives in one place, the society's
ecology.

In [ ]:
REGULATOR_ENV_CONFIG = {
    "ecology_cfg": {"sustainability_weight": 2, "sustainability_threshold": 0.20}
}


def bilevel_config(
    mechanisms,
    *,
    output_dir,
    num_fishers=10,
    horizon=100,
    generations=1000,
    inner_iterations=50,
    evaluation_seeds=3,
):
    # The canonical experiment of config.yaml, with its sizes as arguments.
    regulator = (
        ESConfig()
        .training(
            sigma=0.15,
            mean_lr=0.10,
            sigma_decay=1.0,
            sigma_lr=0.0,
            min_sigma=0.15,
            max_sigma=0.15,
            episodes=generations,
        )
        .agents(
            AgentConfig(
                id=REGULATOR_ID, policy_id="quota_policy", mechanisms=mechanisms
            )
        )
        .environment(
            horizon=1, env=FisheryRegulatorEnv, env_config=REGULATOR_ENV_CONFIG
        )
        .debugging(seed=42)
        .reporting(queries=ES_QUERIES)
    )
    society = (
        APPOptimizerConfig()
        .resources(num_cpus_for_main_process=1)
        .framework(framework="torch")
        .api_stack(
            enable_rl_module_and_learner=True, enable_env_runner_and_connector_v2=True
        )
        .environment(
            env=FisheryRegulatedEnv,
            env_config={
                "ecology_cfg": {
                    "r": 0.3,
                    "K": 5000,
                    "p": 1.0,
                    "B0": 4000,
                    "fish_init": 4000,
                    "sigma": 0.02,
                    "initial_stock_log_sigma": 0.05,
                    "unregulated_f_multiplier": 2.0,
                },
                "seed": 0,
            },
            horizon=horizon,
            disable_env_checking=False,
            schema=FisheryMetricSchema,
            queries=(),
        )
        .env_runners(
            num_env_runners=0,
            num_cpus_per_env_runner=1,
            num_gpus_per_env_runner=0,
            num_envs_per_env_runner=4,
            rollout_fragment_length=horizon,
            batch_mode="truncate_episodes",
            max_requests_in_flight_per_env_runner=1,
        )
        .learners(num_learners=0, num_gpus_per_learner=0)
        .callbacks(
            on_episode_created=tag_episode_with_env_idx,
            on_episode_end=log_and_report_episode_metrics,
        )
        .training(
            episodes=inner_iterations,
            vtrace=True,
            circular_buffer_num_batches=4,
            circular_buffer_iterations_per_batch=1,
            broadcast_interval=1,
            timeout_s_sampler_manager=10,
            timeout_s_aggregator_manager=10,
            gamma=0.99,
            lr=0.001,
            train_batch_size_per_learner=horizon,
            minibatch_size=horizon,
            num_epochs=1,
            entropy_coeff=0.001,
            grad_clip=40.0,
        )
        .evaluation(
            evaluation_interval=1,
            evaluation_duration=1,
            evaluation_duration_unit="episodes",
            evaluation_num_env_runners=0,
            evaluation_parallel_to_training=False,
            evaluation_config={
                "explore": False,
                "rollout_fragment_length": horizon,
                "batch_mode": "complete_episodes",
                "max_requests_in_flight_per_env_runner": 1,
            },
            base_seed=42,
            num_seeds=evaluation_seeds,
        )
        .agents(
            FishermanConfig(
                id="fisherman",
                policy_id="fisher_policy",
                shared_policy=True,
                count=num_fishers,
                mechanisms=(
                    FishingConfig(id="harvest", action_space=UNBOUNDED_BOX),
                    RestoreConfig(id="restore", action_space=UNBOUNDED_BOX),
                ),
                observation_space=spaces.Box(-np.inf, np.inf, (5,), np.float32),
            )
        )
        .fault_tolerance(restart_failed_env_runners=False)
        .debugging(seed=42, num_seeds=1)
        .reporting(
            min_time_s_per_iteration=0,
            min_sample_timesteps_per_iteration=0,
            min_train_timesteps_per_iteration=0,
            queries=INNER_QUERIES,
        )
    )
    return (
        BilevelConfig()
        .world(world_name="fishery_world")
        .reporter(config=CSVConfig(project="bilevel", output_dir=output_dir))
        .ray(device="cpu", num_cpus=4, omp_threads=1, logging_level="INFO")
        .regulator(regulator)
        .society(society)
    )


canonical = bilevel_config((full_stack[0],), output_dir=WORK_DIR.name)
# The regulator's environment holds no capacity; only the society's ecology does.
assert "K" not in canonical.outer_cfg.env_config["ecology_cfg"]
assert canonical.inner_cfg.env_config["ecology_cfg"]["K"] == CAPACITY
print(type(canonical).__name__, "built for the canonical quota-only experiment")

The canonical experiment holds the quota alone, ten fishers, an episode of 100
steps and 1000 generations of the outer search, which takes a long time. To run
the builder end to end inside this notebook, the next cell builds a tiny version
with the full stack of five mechanisms, two fishers, an episode of 20 steps and
two generations. `build_optimizer()` starts Ray and creates the two levels, and
`train()` runs the outer search, which trains and evaluates the fishers against
each candidate. A social-influence pair needs the observation entries of section
7, which for the five-entry observation fits exactly two fishers, hence the two
fishers of this run.

In [ ]:
ray.shutdown()
tiny = bilevel_config(
    full_stack,
    output_dir=WORK_DIR.name,
    num_fishers=2,
    horizon=20,
    generations=2,
    inner_iterations=2,
    evaluation_seeds=1,
)
optimizer = tiny.build_optimizer()
try:
    summary = optimizer.train()
finally:
    ray.shutdown()

print("generations run:", summary["episodes"], "| converged:", summary["converged"])
print("best fitness:", summary["best_fitness"])
best = unflatten(optimizer.outer.action_space, summary["best_mechanism"])
print("best candidate:", {name: value.tolist() for name, value in best.items()})
assert summary["episodes"] == 2
assert np.isfinite(summary["best_fitness"])
assert summary["best_mechanism"].shape == (2,)

The summary reports the best candidate as a normalised vector, which the cell
unflattens into one array per mechanism, exactly as section 9 did. With two
generations of two episodes the numbers are only a smoke test, not a result. The
point is that every element of the notebook, the four mechanisms, their
composition in the environment, the search vector and the fitness, runs together
through the present builder.

# 12. Running the full experiment

The canonical experiment is described by `examples/bilevel_fishery/config.yaml`,
whose `experiment` block builds the same `BilevelConfig` with the reporter set to
Weights and Biases, and whose `run` block calls `build_optimizer` and `train`.
Launch it from the repository root with either command:

```bash
metamarl run examples/bilevel_fishery/config.yaml
python -m core.config.cli run examples/bilevel_fishery/config.yaml
```

The command-line entry point also restarts the process with a fixed
`PYTHONHASHSEED`, which makes the order of the agents in RLlib's batches, and
therefore the results of a seeded run, reproducible. A notebook does not do this,
so the tiny run above is not bit-for-bit reproducible. The command `metamarl check`
validates a configuration file without running it. To study another mechanism,
change the `mechanisms` tuple of the regulator in the YAML file, keeping the
searched mechanisms' action spaces in $[0,1]$ and the `acts_on` names equal to
those of the fishers' own mechanisms.

## What to remember

A mechanism is a rule with a mathematical form, a configuration, and three
methods. `decode` and `apply` return a residual of the transition, and `observe`
returns a residual of the observation. The quota cuts the raw action, the subsidy
and the penalty shift the reward, and social influence writes the peers'
delivered actions into the observation. A regulator composes its mechanisms by
adding their residuals, so one quota per channel is the rule, and the outer search
sees only the mechanisms that have a non-empty action space. The fitness of the
regulator is computed on the realised harvest and the biomass of the last steps of
each evaluation episode, which is a different quantity from the reward that the
fishers learn from.

## References

- Hilborn, R., and Walters, C. J. (1992). *Quantitative Fisheries Stock
  Assessment: Choice, Dynamics and Uncertainty*. Chapman and Hall.
- Jaques, N., Lazaridou, A., Hughes, E., Gulcehre, C., Ortega, P., Strouse, D.,
  Leibo, J. Z., and de Freitas, N. (2019). Social influence as intrinsic
  motivation for multi-agent deep reinforcement learning. *Proceedings of the
  36th International Conference on Machine Learning*, PMLR 97:3040-3049.
- Madani, K., and Dinar, A. (2013). Exogenous regulatory institutions for
  sustainable common pool resource management: Application to groundwater.
  *Water Resources and Economics*, 2-3, 57-76.
  https://doi.org/10.1016/j.wre.2013.08.001
- Pella, J. J., and Tomlinson, P. K. (1969). A generalized stock production model.
  *Bulletin of the Inter-American Tropical Tuna Commission*, 13(3), 421-458.
- Salimans, T., Ho, J., Chen, X., Sidor, S., and Sutskever, I. (2017). Evolution strategies as
  a scalable alternative to reinforcement learning. arXiv:1703.03864.
- Schaefer, M. B. (1954). Some aspects of the dynamics of populations important to
  the management of the commercial marine fisheries. *Bulletin of the
  Inter-American Tropical Tuna Commission*, 1(2), 27-56.
- Yang, J., Wang, E., Trivedi, R., Zhao, T., and Zha, H. (2022). Adaptive
  incentive design with multi-agent meta-gradient reinforcement learning.
  *Proceedings of the 21st International Conference on Autonomous Agents and
  Multiagent Systems (AAMAS)*. arXiv:2112.10859.